In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## lite

In [ ]:
"""--------------------------------------------"""
# [v] make sure that encoding weights make sense
# cvr2 and delta r2, oop-ify
# add movement (average normalized me on a trial)
"""--------------------------------------------"""
# add time
# one regressor

In [ ]:
from core.data import load_sess

# get data
(spike_times, trial_data, psths, session_data, regions) = load_sess(
    subj_id=subj_id,
    sess_id=sess_id,
    tpre=0.5,
    tpost=1,
    alignment="choice",
    tpre_ref=0.5,
    tpost_ref=1,
    alignment_ref="choice",
    binwidth_ms=25,
    thresh=1,
)

In [ ]:
from sg.fitlvm_utils import get_data_model
import numpy as np

# make robs
# when constructing the design matrix, add the idx as a drift term
(data_gd, train_dl, val_dl, test_dl, indices, num_trials, num_tv, num_units) = (
    get_data_model(
        psths,
        trial_data,
        strategy_filter=None,
        regions=regions,
        norm=True,
        num_tents=5,
        task_vars=[
            "response",
            "rewarded",
            "block_side",
            "response_prev",
            "rewarded_prev",
        ],
        sanity_check=0,
    )
)
sample = data_gd[:]
robs = sample["robs"].detach().cpu().numpy()
tvs = np.asarray(sample["tv"].detach().cpu().numpy())
tents = sample["tents"].detach().cpu().numpy()

In [ ]:
from sklearn.preprocessing import OneHotEncoder as OHE

task_vars = ["response", "rewarded", "block_side", "response_prev", "rewarded_prev"]

ohe = OHE().fit(trial_data[task_vars])
tv_names = np.concatenate(
    ([f"tents_{i}" for i in range(tents.shape[1])], ohe.get_feature_names_out())
)

In [ ]:
from sklearn.linear_model import RidgeCV

dm = np.hstack((tents, tvs))

baseline_model = RidgeCV(
    alphas=np.logspace(-5, 5, 11, base=10),
    alpha_per_target=True,
).fit(tents, robs)

robs_predict_baseline = baseline_model.predict(tents)

encoder = RidgeCV(
    alphas=np.logspace(-5, 5, 11, base=10),
    alpha_per_target=True,
).fit(dm, robs)

robs_predict = encoder.predict(dm)

In [ ]:
plt.figure()
plt.imshow(dm, aspect="auto")
plt.show()

In [ ]:
unit_idx = 10
plt.figure(figsize=(2, 4))
plt.imshow(encoder.coef_[10].reshape(-1, 1), vmin=-2, vmax=2, cmap="coolwarm")
plt.xticks([])
plt.yticks(np.arange(encoder.coef_.shape[1]), tv_names)
plt.colorbar(label=r"$\beta$ weight")
plt.show()

In [ ]:
def get_tavg_sc_cond(robs, trial_data, cond):
    if cond == "response":
        left_mask = trial_data.response == 1
        right_mask = trial_data.response == -1

        sc_tavg = {
            "left": robs[left_mask].mean(axis=0),
            "right": robs[right_mask].mean(axis=0),
        }

    return sc_tavg

In [ ]:
sc_tavg = get_tavg_sc_cond(robs - robs_predict_baseline, trial_data, cond="response")

In [ ]:
keys = sc_tavg.keys()
idx = 0
plt.figure()
plt.bar(keys, [sc_tavg[k][idx] for k in keys])
plt.show()

In [ ]:
# check the weights without fitting the drift term
from squiggs.renderers import PETHWeightRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR
from core.data import get_psths_cond, get_choice_ts

reg = "DMS"
mode = "response"

r = PETHWeightRenderer(
    weights=encoder.coef_,
    weight_names=tv_names,
    robs=robs,
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(trial_data, mode=mode),
    spike_times=spike_times[reg],
    peths=get_psths_cond(psths[reg], trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(num_units=num_units, render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
plt.figure(tight_layout=True)
plt.scatter(sc_tavg["right"], encoder.coef_[:, 5])
plt.xlabel("norm sc, right trials")
plt.ylabel("encoder weight, right choice")
plt.show()

In [ ]:
plt.figure(tight_layout=True)
plt.scatter(sc_tavg["left"], encoder.coef_[:, 6])
plt.xlabel("norm sc, left trials")
plt.ylabel("encoder weight, left choice")
plt.show()

In [ ]:
# make sure that encoding weights make sense
# plot weights per unit and label, then transition to renderer.

plt.figure()
plt.imshow(encoder.coef_, aspect="auto", vmin=-2, vmax=2, cmap="coolwarm")
plt.colorbar()
plt.show()